In [ ]:
import requests
import os
import dotenv

dotenv.load_dotenv()

BASE_URL = 'http://apis.data.go.kr/6280000/incheon-road-hazard/hazard-list'
SERVICE_KEY = os.getenv('HARZARD_API_KEY')
MAX_PAGE_SIZE = 3000
print(f"[INFO] base_url={BASE_URL}, serviceKey={SERVICE_KEY[:5]}..., MAX_PAGE_SIZE={MAX_PAGE_SIZE}")


In [ ]:
import pandas as pd

def get_trafic_volume_json(YMD, pageNo=1, numOfRows=MAX_PAGE_SIZE):
    url = BASE_URL + f"?serviceKey={SERVICE_KEY}&pageNo={pageNo}&numOfRows={numOfRows}&YMD={YMD}"
    return requests.get(url).json()

def extract_items(response_json):
    items = response_json["response"]["body"]["items"]
    processed = []
    for item in items:
        new_item = {}
        for k, v in item.items():
            if k.startswith("hour"):
                new_item[k] = int(v)
            else:
                new_item[k] = v
        processed.append(new_item)
    return processed

json_data = get_trafic_volume_json('20260101')
items = extract_items(json_data)
len(items)

In [ ]:
import mysql.connector as mc 
from mysql.connector import Error
import dotenv
import os
from datetime import datetime, timedelta

dotenv.load_dotenv()

DB_CONFIG = {
        'host': os.getenv('DB_HOST'),
        'port': int(os.getenv('DB_PORT', 3306)),
        'user': os.getenv('DB_USER'),
        'password': os.getenv('DB_PASSWORD'),
        'database': os.getenv('DB_DATABASE'),
        'autocommit': False,
}
print_config = {key: DB_CONFIG[key] if key != 'password' else '****' for key in DB_CONFIG}
print(f"[INFO] DB_CONFIG={print_config}")

# statDate, roadName, linkID, direction, startName, endName, hour01 ... hour23
columns = [
    "statDate", "roadName", "linkID", "direction",
    "startName", "endName"
] + [f"hour{str(i).zfill(2)}" for i in range(24)]
col_str = ", ".join(columns)

# %s, %s, ...
placeholders = ", ".join(["%s"] * len(columns))

insert_query = f"""
INSERT INTO traffic_raw ({col_str})
VALUES ({placeholders})
"""
insert_query

In [ ]:
# 테이블 없거나 생성 쿼리가 변경 되었을 때만 실행!!

create_query = """
CREATE TABLE IF NOT EXISTS traffic_raw (
    id BIGINT AUTO_INCREMENT PRIMARY KEY,

    statDate DATE NOT NULL,
    roadName VARCHAR(100),
    linkID VARCHAR(20) NOT NULL,
    direction VARCHAR(10) NOT NULL,
    startName VARCHAR(100),
    endName VARCHAR(100),

    hour00 INT, hour01 INT, hour02 INT,
    hour03 INT, hour04 INT, hour05 INT,
    hour06 INT, hour07 INT, hour08 INT,
    hour09 INT, hour10 INT, hour11 INT,
    hour12 INT, hour13 INT, hour14 INT,
    hour15 INT, hour16 INT, hour17 INT,
    hour18 INT, hour19 INT, hour20 INT,
    hour21 INT, hour22 INT, hour23 INT,

    UNIQUE KEY uniq_traffic (statDate, linkID, direction)
);
"""

conn = mc.connect(**DB_CONFIG)
cursor = conn.cursor()

try:
    cursor.execute(create_query)
    conn.commit()
    print("[INFO] traffic_raw 테이블 생성 완료")
except Exception as e:
    print(f"[ERROR] 테이블 생성 실패: {e}")
    conn.rollback()

cursor.close()
conn.close()

In [ ]:
import mysql.connector as mc
from mysql.connector import Error

conn = mc.connect(**DB_CONFIG)
cursor = conn.cursor()

def generate_dates(year, m=1, d=1):
    start = datetime(year, m, d)
    end = datetime(year, 12, 31)
    while start <= end:
        yield start.strftime("%Y%m%d")
        start += timedelta(days=1)

def dict_to_tuple(item):
    return tuple(item[col] for col in columns)

total_inserted = 0

for date in generate_dates(2025):
    print(f"[INFO] Processing {date}")

    try:
        # 1️⃣ API 호출
        json_data = get_trafic_volume_json(date)

        # 2️⃣ 데이터 추출
        items = extract_items(json_data)

        if not items:
            print(f"[WARN] No data for {date}")
            continue

        # 3️⃣ dict → tuple 변환
        values = [dict_to_tuple(item) for item in items]

        # 4️⃣ insert
        cursor.executemany(insert_query, values)

        # 5️⃣ commit
        conn.commit()

        total_inserted += len(values)
        print(f"[INFO] Inserted {len(values)} rows")

    except Exception as e:
        print(f"[ERROR] {date}: {e}")
        conn.rollback()

cursor.close()
conn.close()

print(f"[DONE] Total inserted: {total_inserted}")